In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
image_train_folder = "/content/drive/MyDrive/house-recognition/train"
img_dir = "/content/drive/MyDrive/house-recognition/train/train"


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, debug=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.debug = debug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = row['image_name']
        img_path = os.path.join(self.img_dir, img_name)

        # debug path
        if self.debug and idx < 3:
            print(f"[DEBUG] img_name: {img_name}")
            print(f"[DEBUG] img_path: {img_path}")
            print(f"[DEBUG] exists: {os.path.exists(img_path)}")

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"File not found: {img_path}")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # train
        if 'class' in self.df.columns:
            label = row['class']

            if self.debug and idx < 3:
                print(f"[DEBUG] label: {label}")
                print(f"[DEBUG] image shape: {image.shape}")

            return image, label

        # test
        else:
            return image, row['image_name']

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/house-recognition/train.csv")

full_dataset = HouseDataset(train_df, "/content/drive/MyDrive/house-recognition/train/train", transform=train_transform)

# เปิด debug แค่ train ก่อน
full_dataset = HouseDataset(train_df, img_dir, transform=train_transform, debug=True)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform
val_dataset.dataset.debug = False  # ปิด debug ใน val

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torchvision.models as models
import torch.nn as nn
from torchvision.models import EfficientNet_B0_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

model = model.to(device)

In [ ]:
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # ลด LR
EPOCHS = 10

In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        # debug batch แรก
        if batch_idx == 0:
            print(f"[DEBUG] batch images shape: {images.shape}")
            print(f"[DEBUG] batch labels shape: {labels.shape}")
            print(f"[DEBUG] labels: {labels[:5]}")

        optimizer.zero_grad()
        outputs = model(images)

        # debug output
        if batch_idx == 0:
            print(f"[DEBUG] outputs shape: {outputs.shape}")
            print(f"[DEBUG] outputs sample: {outputs[:2]}")

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            # debug
            if batch_idx == 0:
                print(f"[DEBUG] preds: {preds[:10]}")
                print(f"[DEBUG] labels: {labels[:10]}")

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [ ]:
EPOCHS = 15

best_acc = 0

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1} =====")

    loss = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)

    print(f"Loss={loss:.4f}, Val Acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")


===== Epoch 1 =====
[DEBUG] batch images shape: torch.Size([32, 3, 224, 224])
[DEBUG] batch labels shape: torch.Size([32])
[DEBUG] labels: tensor([0, 1, 0, 0, 1], device='cuda:0')
[DEBUG] outputs shape: torch.Size([32, 2])
[DEBUG] outputs sample: tensor([[ 0.8050, -1.1896],
        [-1.6885,  1.9836]], device='cuda:0', grad_fn=<SliceBackward0>)
[DEBUG] preds: tensor([0, 1, 1, 0, 1, 0, 0, 1, 0, 0], device='cuda:0')
[DEBUG] labels: tensor([0, 1, 1, 0, 1, 0, 0, 1, 0, 0], device='cuda:0')
Loss=0.1954, Val Acc=0.9408

===== Epoch 2 =====
[DEBUG] batch images shape: torch.Size([32, 3, 224, 224])
[DEBUG] batch labels shape: torch.Size([32])
[DEBUG] labels: tensor([0, 0, 1, 1, 0], device='cuda:0')
[DEBUG] outputs shape: torch.Size([32, 2])
[DEBUG] outputs sample: tensor([[ 2.4292, -2.3145],
        [ 0.6781, -0.2945]], device='cuda:0', grad_fn=<SliceBackward0>)
[DEBUG] preds: tensor([0, 1, 1, 0, 1, 0, 0, 1, 0, 0], device='cuda:0')
[DEBUG] labels: tensor([0, 1, 1, 0, 1, 0, 0, 1, 0, 0], device=

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch

# 🔧 path (แก้ถ้าจำเป็น)
test_dir = "/content/drive/MyDrive/house-recognition/test/test"

# ✅ 1. สร้าง test_df 
test_files = sorted(os.listdir(test_dir))

test_df = pd.DataFrame({
    "image_name": test_files
})

print("Total test images:", len(test_df))

import torchvision.models as models
import torch.nn as nn

model1 = models.resnet18(pretrained=False)
model1.fc = nn.Linear(model1.fc.in_features, 2)

model2 = models.efficientnet_b0(pretrained=False)
model2.classifier[1] = nn.Linear(model2.classifier[1].in_features, 2)


# ✅ 2. โหลด model ที่ดีที่สุด
model1.load_state_dict(torch.load("best_model.pth"))
model2.load_state_dict(torch.load("best_model_effnet.pth"))

model1.to(device)
model2.to(device)

model1.eval()
model2.eval()

mean = [0.485,0.456,0.406]
std = [0.229,0.224,0.225]

# ✅ 3. TTA transforms
tta_transforms = [
    val_transform,

    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
]

# ✅ 4. Predict
predictions = []

with torch.no_grad():
    for idx in tqdm(range(len(test_df))):
        img_name = test_df.iloc[idx]['image_name']
        img_path = os.path.join(test_dir, img_name)

        image = Image.open(img_path).convert("RGB")

        outputs = []

        for t in tta_transforms:
            img = t(image).unsqueeze(0).to(device)

            out1 = model1(img)
            out2 = model2(img)

            # ใช้ softmax ก่อน ensemble 
            out = 0.6 * out2 + 0.4 * out1
            out = torch.softmax(out, dim=1)
            outputs.append(out)

        avg_output = torch.mean(torch.stack(outputs), dim=0)
        pred = torch.argmax(avg_output, dim=1).item()

        predictions.append(pred)

# ✅ 5. สร้าง submission
submission = pd.DataFrame({
    "id": test_df["image_name"].str.replace(".jpg", "", regex=False),
    "answer": predictions
})

submission.to_csv("submission.csv", index=False)

print("✅ Saved submission.csv")

Total test images: 1550


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


RuntimeError: Error(s) in loading state_dict for ResNet:
	Missing key(s) in state_dict: "conv1.weight", "bn1.weight", "bn1.bias", "bn1.running_mean", "bn1.running_var", "layer1.0.conv1.weight", "layer1.0.bn1.weight", "layer1.0.bn1.bias", "layer1.0.bn1.running_mean", "layer1.0.bn1.running_var", "layer1.0.conv2.weight", "layer1.0.bn2.weight", "layer1.0.bn2.bias", "layer1.0.bn2.running_mean", "layer1.0.bn2.running_var", "layer1.1.conv1.weight", "layer1.1.bn1.weight", "layer1.1.bn1.bias", "layer1.1.bn1.running_mean", "layer1.1.bn1.running_var", "layer1.1.conv2.weight", "layer1.1.bn2.weight", "layer1.1.bn2.bias", "layer1.1.bn2.running_mean", "layer1.1.bn2.running_var", "layer2.0.conv1.weight", "layer2.0.bn1.weight", "layer2.0.bn1.bias", "layer2.0.bn1.running_mean", "layer2.0.bn1.running_var", "layer2.0.conv2.weight", "layer2.0.bn2.weight", "layer2.0.bn2.bias", "layer2.0.bn2.running_mean", "layer2.0.bn2.running_var", "layer2.0.downsample.0.weight", "layer2.0.downsample.1.weight", "layer2.0.d

In [ ]:
from google.colab import files
files.download("submission.csv")